# Model Development Visualization

This notebook compares four model-development steps:

1. Baseline ResNet-50 using cropped SEM images only
2. Main multimodal model using cropped SEM images plus microscope metadata
3. Hyperparameter experiment using a lower learning rate and more epochs
4. Backbone unfreeze experiment using full fine-tuning from the beginning

The goal is to show how each modeling decision influenced validation accuracy and loss.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
outputs_dir = Path(r"C:/Users/kom-e14-1/Documents/Codex/2026-06-05/i-have-sem-images-i-chategorized/outputs")

experiments = {
    "Baseline ResNet-50\nLR=3e-5, 20 epochs": outputs_dir / "sem_resnet_model",
    "Image + Metadata\nLR=2e-5, 20 epochs": outputs_dir / "sem_multimodal_resnet_model",
    "Hyperparameter Trial\nLR=1e-5, 30 epochs": outputs_dir / "sem_resnet_model_lr1e5_epoch30",
    "Unfreeze From Start\nLR=3e-5, 20 epochs": outputs_dir / "sem_resnet_model_unfreeze0",
}

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

## Step 1: Baseline ResNet-50

The first model used pretrained `microsoft/resnet-50` and cropped SEM images only. This is the baseline because it gives a simple but strong reference model before adding metadata or tuning hyperparameters.

## Step 2: Add Metadata

The second model used the same image branch but added microscope metadata features extracted from the white ribbon. The metadata branch received numeric features such as `eht_kv`, `wd_mm`, `mag_value`, and categorical features such as `signal` and `mag_unit`.

## Step 3: Hyperparameter Trial

The third experiment returned to the image-only ResNet-50 model but changed hyperparameters: the learning rate was reduced from `3e-5` to `1e-5`, and the number of epochs increased from 20 to 30. This tested whether slower fine-tuning would improve validation performance.

## Step 4: Unfreeze Backbone From Start

The fourth experiment kept the successful baseline learning rate `3e-5` and 20 epochs, but changed `freeze-backbone-epochs` from 3 to 0. This tested whether full fine-tuning from the beginning would help the pretrained ResNet-50 adapt faster to SEM images.

In [ ]:
histories = []
summary_rows = []

for name, folder in experiments.items():
    history = pd.DataFrame(load_json(folder / "history.json"))
    history["experiment"] = name
    histories.append(history)

    metrics = load_json(folder / "metrics.json")
    report = metrics["classification_report"]
    summary_rows.append({
        "experiment": name,
        "best_val_accuracy": metrics["best_val_accuracy"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "macro_f1": report["macro avg"]["f1-score"],
    })

histories = pd.concat(histories, ignore_index=True)
summary = pd.DataFrame(summary_rows).sort_values("best_val_accuracy", ascending=False)
summary

## Accuracy by Epoch

This is the most common graph for a classification training report. It shows whether validation performance improves, plateaus, or decreases over time.

In [ ]:
plt.figure(figsize=(10, 5))
for experiment, group in histories.groupby("experiment"):
    plt.plot(group["epoch"], group["val_accuracy"], marker="o", label=experiment)
plt.title("Validation Accuracy During Model Development")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Loss by Epoch

Loss curves help show whether the model is still learning and whether validation loss becomes unstable.

In [ ]:
plt.figure(figsize=(10, 5))
for experiment, group in histories.groupby("experiment"):
    plt.plot(group["epoch"], group["train_loss"], linestyle="--", label=f"{experiment} train")
    plt.plot(group["epoch"], group["val_loss"], marker="o", label=f"{experiment} val")
plt.title("Training and Validation Loss During Model Development")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Final Comparison

A bar chart is useful for the final report because it compares the best validation accuracy of each experiment directly.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(summary["experiment"], summary["best_val_accuracy"])
plt.title("Best Validation Accuracy by Experiment")
plt.xlabel("")
plt.ylabel("Best validation accuracy")
plt.ylim(0.85, 0.95)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

summary

## Interpretation

The baseline ResNet-50 image-only model achieved the best validation accuracy at approximately 92.97%. The unfreeze-from-start experiment reached the same validation accuracy, so changing the freeze schedule did not improve the model. Adding metadata produced 92.43%, which was slightly lower, likely because OCR metadata was noisy and incomplete. The lower-learning-rate hyperparameter trial reached 90.81%, so it did not improve the baseline.

For this project, the baseline image-only ResNet-50 remains the best model. Future improvement should focus on collecting more real labeled data for minority classes and clarifying confusing class boundaries, especially between `close_up_line` and `Electrode`.